# Bandit feedback sensitivity — sgd_one_class_svm / r3_full_causal_identity
Same score stream and all three feedback branches; seed 42 screen and seeds 42–44 for finalists.

In [ ]:
import base64, gzip, json, sys
from pathlib import Path
bundle=json.loads(gzip.decompress(base64.b64decode("...")))
pkg=Path("/kaggle/working/fraudlab_pkg/fraudlab"); pkg.mkdir(parents=True,exist_ok=True)
for name,source in bundle.items(): (pkg/name).write_text(source,encoding="utf-8")
sys.path.insert(0,str(pkg.parent))


In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from fraudlab.config import ExperimentConfig, FeedbackConfig, BanditConfig
from fraudlab.data import chronological_positions
from fraudlab.features import load_feature_artifact
from fraudlab.simulator import run_bandit_simulation, run_threshold_baselines, save_simulation_outputs
from fraudlab.metrics import cost_summary
MODEL_NAME="sgd_one_class_svm"
RECIPE="r3_full_causal_identity"
SEEDS=[42]
root=Path("/kaggle/input")
feature_files=list(root.rglob("ieee_cis_features.parquet"))
tx=[p for p in feature_files if p.parent.name=="transaction"]
ident=[p for p in feature_files if p.parent.name=="identity"]
feature_path=(ident[0] if RECIPE=="r3_full_causal_identity" else tx[0])
pred_files=list(root.rglob(MODEL_NAME+"_simulation_predictions.parquet"))
threshold_files=list(root.rglob(MODEL_NAME+"_thresholds.json"))
if not pred_files or not threshold_files: raise FileNotFoundError("Attach the matching training notebook outputs")
config=ExperimentConfig()
frame,metadata=load_feature_artifact(feature_path)
pos=chronological_positions(frame,config.split)["simulation"]
rows=frame.iloc[pos].reset_index(drop=True)
pred=pd.read_parquet(pred_files[0]).reset_index(drop=True)
if len(pred)!=len(rows): raise ValueError("Prediction length differs from held-out stream")
if "TransactionID" in pred and not np.array_equal(pred.TransactionID.to_numpy(),rows.TransactionID.to_numpy()): raise ValueError("Prediction order mismatch")
prob=pred.probability.to_numpy(dtype=float)
threshold=json.loads(threshold_files[0].read_text(encoding="utf-8"))
out=Path("/kaggle/working/outputs/bandit")/(MODEL_NAME+"__"+RECIPE)
out.mkdir(parents=True,exist_ok=True)
records=[]
for seed in tqdm(SEEDS,desc="Bandit seeds",unit="seed"):
  oracle_dec,oracle=run_bandit_simulation(rows,prob,"oracle",config.costs,FeedbackConfig(seed=seed),BanditConfig(seed=seed),seed)
  oracle["branch"]="immediate_feedback_oracle"; oracle["model"]=MODEL_NAME; oracle["recipe"]=RECIPE; oracle["oracle_cost"]=oracle["total_cost"]; oracle["oracle_tax"]=0.0
  records.append(oracle); save_simulation_outputs(oracle_dec,oracle,out,f"seed_{seed}_oracle")
  for branch,delay in [("blocked_hidden",None),("blocked_after_14d",14)]:
    feedback=FeedbackConfig(blocked_label_delay_days=delay,seed=seed)
    decisions,metrics=run_bandit_simulation(rows,prob,"closed_loop",config.costs,feedback,BanditConfig(seed=seed),seed)
    metrics.update({"branch":branch,"model":MODEL_NAME,"recipe":RECIPE,"oracle_cost":oracle["total_cost"],"oracle_tax":metrics["total_cost"]-oracle["total_cost"]})
    records.append(metrics); save_simulation_outputs(decisions,metrics,out,f"seed_{seed}_{branch}")
    if metrics["review_count"]>metrics["review_cap_count"]: raise AssertionError("Daily review cap was exceeded")
  baselines=run_threshold_baselines(rows,prob,config.costs,threshold)
  (out/("seed_"+str(seed)+"_baselines.json")).write_text(json.dumps(baselines,indent=2),encoding="utf-8")
metrics=pd.DataFrame(records)
metrics.to_csv(out/"metrics.csv",index=False)
summary=metrics.groupby(["model","recipe","branch"],dropna=False).agg({"total_cost":["mean","std"],"oracle_tax":["mean","std"],"pr_auc":["mean","std"],"review_rate":["mean","std"],"feedback_updates":["mean","std"]}).reset_index()
summary.to_csv(out/"metrics_summary.csv",index=False)
plt.figure(figsize=(11,5)); plt.bar(metrics.branch,metrics.total_cost); plt.xticks(rotation=20,ha="right"); plt.ylabel("Decision cost"); plt.title(MODEL_NAME+" / "+RECIPE+" — feedback sensitivity"); plt.tight_layout(); plt.savefig(out/"feedback_costs.png",dpi=150); plt.close()
print(metrics[["model","recipe","branch","seed","total_cost","oracle_cost","oracle_tax","review_count","approve_count","block_count","feedback_updates","pr_auc","precision_at_0_5","recall_at_0_5"]].to_string(index=False))
